# apertus-eval-prep — paper-matrix **vLLM backend** (Colab ID-5)

[Open in Colab](https://colab.research.google.com/github/Shivani767/apertus-eval-prep/blob/master/notebooks/colab_stability_backend.ipynb)

Runtime → **T4 GPU**.

Do **not** Run all. Every session: **cell 1 → cell 2 → cell 3 → one sweep**.

Cell 2 installs pinned **vLLM 0.24.0**. Success: `OK vllm 0.24.0 torch 2.11...`

**Paper matrix on GitHub:** 19/34 cells. **All three `backend=vllm` cells are DONE.**

| Status | Cell |
|---|---|
| **DONE** | SmolLM2 `backend=vllm` (336/800) |
| **DONE** | Qwen-3B `backend=vllm` (534/800) |
| **DONE** | Phi `backend=vllm` (537/800) |
| Next work | HF notebook: quant / seed — [`colab_stability.ipynb`](colab_stability.ipynb) |
| Inventory | [`paper/run_status.md`](../paper/run_status.md) |

Drive: `MyDrive/apertus-eval-prep-paper`.


In [ ]:
# Cell 1 — clone / pull
import os
from pathlib import Path

if Path("pyproject.toml").exists() and Path("src/apertus_eval_prep").exists():
    print("Already in repo root")
    !git pull --ff-only
elif Path("apertus-eval-prep/pyproject.toml").exists():
    %cd apertus-eval-prep
    !git pull --ff-only
else:
    !git clone https://github.com/Shivani767/apertus-eval-prep.git
    %cd apertus-eval-prep
!pip -q install -e ".[viz]"
!git log -1 --oneline
print("install script:", Path("scripts/colab_install_vllm.py").resolve())
!grep -n 'VLLM_VER\|VLLM_WHEEL' scripts/colab_install_vllm.py | head


In [ ]:
# Cell 2 — vLLM install (run AFTER cell 1). Live logs; do not skip clone/pull.
import os, sys, subprocess
from pathlib import Path

def _repo_root() -> Path:
    for p in (Path.cwd(), Path("/content/apertus-eval-prep"), Path("apertus-eval-prep")):
        if (p / "pyproject.toml").exists() and (p / "src" / "apertus_eval_prep").exists():
            os.chdir(p)
            return p
    if not Path("/content/apertus-eval-prep").exists():
        subprocess.check_call(["git", "clone", "https://github.com/Shivani767/apertus-eval-prep.git", "/content/apertus-eval-prep"])
    os.chdir("/content/apertus-eval-prep")
    return Path.cwd()

root = _repo_root()
print("cwd:", root, flush=True)

subprocess.check_call(["git", "fetch", "origin"])
subprocess.check_call(["git", "reset", "--hard", "origin/master"])
print(subprocess.check_output(["git", "log", "-1", "--oneline"], text=True), flush=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[viz]"])

import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU, then Runtime → Restart session, then cell 1 → cell 2."
print(torch.cuda.get_device_name(0), "torch", torch.__version__, "cuda", torch.version.cuda, flush=True)

print("install script:", (root / "scripts/colab_install_vllm.py").resolve(), flush=True)
subprocess.check_call(["grep", "-n", "VLLM_VER\\|VLLM_WHEEL", "scripts/colab_install_vllm.py"])

# Stream install. Success line: OK vllm 0.24.0 torch 2.11...
rc = subprocess.call([sys.executable, "-u", "scripts/colab_install_vllm.py"])
if rc != 0:
    raise SystemExit(
        "vLLM install failed. Search this cell for Traceback / SystemExit ABOVE this line. "
        "Then Runtime → Disconnect and delete runtime, reopen, cell 1 → cell 2."
    )

_repo_src = str((root / "src").resolve())
if _repo_src not in sys.path:
    sys.path.insert(0, _repo_src)

if not Path("data/official/eval_set.jsonl").exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[snapshot]"])
    subprocess.check_call([sys.executable, "scripts/snapshot_benchmarks.py"])
else:
    print("official slices already on disk")


In [ ]:
# Cell 3 — Drive + sweep helper
import os, sys
from pathlib import Path
from google.colab import drive, files

def ensure_repo():
    for p in (Path.cwd(), Path("/content/apertus-eval-prep"), Path("apertus-eval-prep")):
        if (p / "pyproject.toml").exists() and (p / "src" / "apertus_eval_prep").exists():
            os.chdir(p)
            src = str((p / "src").resolve())
            if src not in sys.path:
                sys.path.insert(0, src)
            print("cwd:", Path.cwd(), flush=True)
            return p
    raise FileNotFoundError("Repo not found. Run cells 1–2 first.")

ensure_repo()
try:
    import apertus_eval_prep  # noqa: F401
except ModuleNotFoundError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[viz]"])

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/apertus-eval-prep-paper")
DRIVE.mkdir(parents=True, exist_ok=True)
(DRIVE / "runs").mkdir(exist_ok=True)
Path("results/runs").mkdir(parents=True, exist_ok=True)
os.environ["APERTUS_CHECKPOINT_DIR"] = str(DRIVE / "runs")

if (DRIVE / "registry_paper.jsonl").exists():
    !cp -a {DRIVE}/registry_paper.jsonl results/registry_paper.jsonl
    print("restored registry from Drive")
if any((DRIVE / "runs").iterdir()):
    !cp -a {DRIVE}/runs/. results/runs/
    print("restored runs from Drive")
print("partials:", len(list(Path("results/runs").glob("*.partial.jsonl"))), flush=True)

def save_paper():
    import subprocess
    if Path("results/registry_paper.jsonl").exists():
        subprocess.check_call(["cp", "-a", "results/registry_paper.jsonl", str(DRIVE / "registry_paper.jsonl")])
    if Path("results/runs").exists():
        subprocess.check_call(["bash", "-lc", f"cp -a results/runs/. {DRIVE}/runs/"])
    n = len(list(Path("results/runs").glob("*.json")))
    print(f"saved to Drive ({n} run JSON files):", DRIVE, flush=True)
    subprocess.check_call(["zip", "-r", "/tmp/paper_matrix_partial.zip", "results/runs", "results/registry_paper.jsonl"])
    files.download("/tmp/paper_matrix_partial.zip")

def sweep(*extra):
    ensure_repo()
    only_model = only_factor = None
    args = list(extra)
    i = 0
    while i < len(args):
        if args[i] == "--only-model" and i + 1 < len(args):
            only_model = args[i + 1]; i += 2
        elif args[i] == "--only-factor" and i + 1 < len(args):
            only_factor = args[i + 1]; i += 2
        else:
            raise ValueError(f"unknown sweep arg {args[i]!r}")
    from vllm import LLM  # noqa: F401
    from apertus_eval_prep.sweep import execute_sweep
    print(f"sweep model={only_model} factor={only_factor}", flush=True)
    planned = execute_sweep(
        study_path=Path("configs/experiments/stability.yaml"),
        repo_root=Path(".").resolve(),
        out_dir=Path("results/runs"),
        registry_path=Path("results/registry_paper.jsonl"),
        profile="t4",
        only_model=only_model,
        only_factor=only_factor,
    )
    print({"n_cells": len(planned), "n_skip": sum(1 for p in planned if p["skipped"])}, flush=True)
    save_paper()

!python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml --profile t4 --dry-run --only-factor backend --out-dir results/runs --registry results/registry_paper.jsonl | head -n 20


## ID-5 — backend = vLLM (all done on GitHub)

| Status | Sweep |
|---|---|
| **DONE** | SmolLM2 |
| **DONE** | Qwen-3B |
| **DONE** | Phi-3.5 |


In [ ]:
# DONE on GitHub — SmolLM2 backend=vllm (336/800). Skip unless dry-run says run.
sweep("--only-model", "HuggingFaceTB/SmolLM2-1.7B-Instruct", "--only-factor", "backend")


In [ ]:
# DONE on GitHub — Qwen-3B backend=vllm (534/800). Skip unless dry-run says run.
sweep("--only-model", "Qwen/Qwen2.5-3B-Instruct", "--only-factor", "backend")


In [ ]:
# DONE on GitHub — Phi-3.5 backend=vllm (537/800). Skip unless dry-run says run.
sweep("--only-model", "microsoft/Phi-3.5-mini-instruct", "--only-factor", "backend")


Unpack the Drive zip on Mac. Commit `results/runs/*.json` and `results/registry_paper.jsonl`.
